In [ ]:
!uname -a && cat /etc/*release

Linux 9c57d981278c 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [ ]:
!pwd
!ls -la

/content
total 16
drwxr-xr-x 1 root root 4096 Jun  4 13:39 .
drwxr-xr-x 1 root root 4096 Jul  4 16:32 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:39 .config
drwxr-xr-x 1 root root 4096 Jun  4 13:39 sample_data


In [ ]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4400.42
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [ ]:
%%writefile matmul.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        printf("Usage: %s <N>\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    int i, j, k;

    double (*a)[n] = malloc(sizeof(double[n][n]));
    double (*b)[n] = malloc(sizeof(double[n][n]));
    double (*c)[n] = malloc(sizeof(double[n][n]));

    if (!a || !b || !c) {
        printf("Memory allocation error for N=%d\n", n);
        return 1;
    }
    for (i = 0; i < n; i++) {
        for (j = 0; j < n; j++) {
            a[i][j] = 2.0;
            b[i][j] = 3.0;
            c[i][j] = 0.0;
        }
    }

    double start_time = omp_get_wtime();

    for (i = 0; i < n; ++i) {
        for (k = 0; k < n; k++) {
            for (j = 0; j < n; ++j) {
                c[i][j] += a[i][k] * b[k][j];
            }
        }
    }

    double run_time = omp_get_wtime() - start_time;
    printf("Computation time (N=%d): %f seconds\n", n, run_time);

    #ifdef DEBUG
    FILE *f = fopen("mat-res.txt", "w");
    if (!f) {
        perror("fopen error");
        return 1;
    }

    fprintf(f, "%d\n\n", n);

    int print_limit = (n < 1000) ? n : 1000;

    for (int i = 0; i < print_limit; i++) {
        for (int j = 0; j < print_limit; j++) {
            fprintf(f, "%.0f ", c[i][j]);
        }
        fprintf(f, "\n");
    }
    fclose(f);
    printf("Debug file 'mat-res.txt' successfully saved.\n");
    #endif
    free(a);
    free(b);
    free(c);

    return 0;
}

Writing matmul.c


In [ ]:
!gcc -O3 -march=native -fopenmp matmul.c -o matmul_seq


In [ ]:
!time ./matmul_seq 5000

Computation time (N=5000): 87.653122 seconds

real	1m27.954s
user	1m26.966s
sys	0m0.345s


In [ ]:
!nvidia-smi

Sat Jul  4 14:20:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             42W /   70W |       0MiB /  15360MiB |     13%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
device_query_str = '''
#include <stdio.h>

int main() {
    int deviceCount = 0;
    cudaGetDeviceCount(&deviceCount);

    if (deviceCount == 0) {
        printf("Nessun dispositivo CUDA trovato.\\n");
        return 1;
    }

    for (int dev = 0; dev < deviceCount; ++dev) {
        cudaDeviceProp deviceProp;
        cudaGetDeviceProperties(&deviceProp, dev);

        printf("\\n--- Device %d: \\"%s\\" ---\\n", dev, deviceProp.name);
        printf("  CUDA Capability Major/Minor version:          %d.%d\\n", deviceProp.major, deviceProp.minor);
        printf("  Total amount of global memory:                %.2f GBytes\\n", (float)deviceProp.totalGlobalMem / (1024 * 1024 * 1024));
        printf("  Number of multiprocessors (SMs):              %d\\n", deviceProp.multiProcessorCount);
        printf("  Total amount of constant memory:              %lu bytes\\n", deviceProp.totalConstMem);
        printf("  Total amount of shared memory per block:      %lu bytes\\n", deviceProp.sharedMemPerBlock);
        printf("  Total number of registers available per block: %d\\n", deviceProp.regsPerBlock);
        printf("  Warp size:                                    %d\\n", deviceProp.warpSize);
        printf("  Maximum threads per multiprocessor (SM):      %d\\n", deviceProp.maxThreadsPerMultiProcessor);
        printf("  Maximum threads per block:                    %d\\n", deviceProp.maxThreadsPerBlock);
        printf("  Max dimension size of a thread block (x,y,z): %d, %d, %d\\n", deviceProp.maxThreadsDim[0], deviceProp.maxThreadsDim[1], deviceProp.maxThreadsDim[2]);
        printf("  Max dimension size of a grid size    (x,y,z): %d, %d, %d\\n", deviceProp.maxGridSize[0], deviceProp.maxGridSize[1], deviceProp.maxGridSize[2]);
    }
    return 0;
}
'''
with open('deviceQuery.cu', 'w') as f:
    f.write(device_query_str)

In [ ]:
!nvcc -arch=sm_75 deviceQuery.cu -o deviceQuery && ./deviceQuery

cc1plus: fatal error: deviceQuery.cu: No such file or directory
compilation terminated.


In [1]:
%%writefile matmul_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>


__global__ void matMulKernel(double *a, double *b, double *c, int n) {

    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // Check that the thread is within the matrix boundaries
    if (row < n && col < n) {
        double sum = 0.0;

        // 1D array simulating a 2D matrix (row * n + col)
        for (int k = 0; k < n; k++) {
            sum += a[row * n + k] * b[k * n + col];
        }
        c[row * n + col] = sum;
    }
}


int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N. Optional: use -d or --debug for output.\n");
        printf("Usage: %s <N> [-d]\n", argv[0]);
        return 1;
    }

    int n = 0;
    int debug_mode = 0;

    // Parse command line arguments dynamically
    for (int i = 1; i < argc; i++) {
        if (strcmp(argv[i], "-d") == 0 || strcmp(argv[i], "--debug") == 0) {
            debug_mode = 1;
        } else {
            n = atoi(argv[i]);
        }
    }

    if (n <= 0) {
        printf("Error: Invalid dimension N.\n");
        return 1;
    }
    size_t size = (size_t)n * n * sizeof(double);

    // I use "flattened" 1D arrays because they are the safest for GPU transfers
    double *h_a = (double *)malloc(size);
    double *h_b = (double *)malloc(size);
    double *h_c = (double *)malloc(size);

    if (!h_a || !h_b || !h_c) {
        printf("Host RAM allocation error for N=%d\n", n);
        return 1;
    }

    // data initialization on the CPU
    for (int i = 0; i < n * n; i++) {
        h_a[i] = 2.0;
        h_b[i] = 3.0;
        h_c[i] = 0.0;
    }

    //cuda timers
    cudaEvent_t start, stop_h2d, stop_kernel, stop_d2h;
    cudaEventCreate(&start);
    cudaEventCreate(&stop_h2d);
    cudaEventCreate(&stop_kernel);
    cudaEventCreate(&stop_d2h);

    // device allocation
    double *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    cudaEventRecord(start); // start the timer

    //data transfer from host to device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    cudaEventRecord(stop_h2d); // end timer of H2D

    //block define
    dim3 threadsPerBlock(16, 16);
    //entire grid of blocks
    dim3 numBlocks((n + 15) / 16, (n + 15) / 16);

    matMulKernel<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaEventRecord(stop_kernel); // end timer of GPU computation

    //data transfer from device to host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop_d2h); // end timer of D2H
    cudaEventSynchronize(stop_d2h); // Wait for everything to completely finish

    float ms_h2d, ms_kernel, ms_d2h;
    cudaEventElapsedTime(&ms_h2d, start, stop_h2d);
    cudaEventElapsedTime(&ms_kernel, stop_h2d, stop_kernel);
    cudaEventElapsedTime(&ms_d2h, stop_kernel, stop_d2h);

    printf("Matrix Dimension N           : %d\n", n);
    printf("Transfer Time CPU -> GPU     : %f ms\n", ms_h2d);
    printf("GPU Kernel Computation Time  : %f ms\n", ms_kernel);
    printf("Transfer Time GPU -> CPU     : %f ms\n", ms_d2h);
    printf("Total Time (Data + Compute)  : %f ms\n", ms_h2d + ms_kernel + ms_d2h);

    // debug section
    if (debug_mode) {
        FILE *debug_file = fopen("debug_c.txt", "w");
        if (debug_file != NULL) {
            // Determine the print size, maximum 10x10 block
            int print_size = (n < 10) ? n : 10;

            fprintf(debug_file, "Debug Matrix C (N=%d)\n", n);
            fprintf(debug_file, "First %dx%d elements (top-left):\n", print_size, print_size);
            fprintf(debug_file, "--------------------------------------------------\n");

            for (int i = 0; i < print_size; i++) {
                for (int j = 0; j < print_size; j++) {
                    // %.2f prints 2 decimal places
                    fprintf(debug_file, "%10.2f ", h_c[i * n + j]);
                }
                fprintf(debug_file, "\n");
            }

            // Print the very last element to ensure the whole kernel worked
            fprintf(debug_file, "\nLast element C[%d][%d] = %.2f\n", n-1, n-1, h_c[(n-1)*n + (n-1)]);

            fclose(debug_file);
            printf("Debug ON: Output successfully saved to 'debug_c.txt'.\n");
        } else {
            printf("Debug ON: Error opening the debug file!\n");
        }
    }
    // ----------------------------------------------------------
    //memory cleanup
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    cudaEventDestroy(start); cudaEventDestroy(stop_h2d);
    cudaEventDestroy(stop_kernel); cudaEventDestroy(stop_d2h);

    return 0;
}

Writing matmul_cuda.cu


In [2]:
!nvcc -arch=sm_75 -O3 matmul_cuda.cu -o matmul_gpu

In [ ]:
!/content/matmul_gpu 5000

Matrix Dimension N           : 5000
Transfer Time CPU -> GPU     : 85.669121 ms
GPU Kernel Computation Time  : 1223.180176 ms
Transfer Time GPU -> CPU     : 50.634144 ms
Total Time (Data + Compute)  : 1359.483398 ms


In [ ]:
!nvprof /content/matmul_gpu 15000

==2840== NVPROF is profiling process 2840, command: /content/matmul_gpu 15000
Matrix Dimension N           : 15000
Transfer Time CPU -> GPU     : 782.544128 ms
GPU Kernel Computation Time  : 38226.054688 ms
Transfer Time GPU -> CPU     : 386.780457 ms
Total Time (Data + Compute)  : 39395.378906 ms
==2840== Profiling application: /content/matmul_gpu 15000
==2840== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   97.03%  38.2254s         1  38.2254s  38.2254s  38.2254s  matMulKernel(double*, double*, double*, int)
                    1.99%  782.03ms         2  391.02ms  387.23ms  394.80ms  [CUDA memcpy HtoD]
                    0.98%  386.33ms         1  386.33ms  386.33ms  386.33ms  [CUDA memcpy DtoH]
      API calls:   99.54%  39.3946s         3  13.1315s  387.47ms  38.6121s  cudaMemcpy
                    0.44%  174.15ms         4  43.538ms     464ns  174.15ms  cudaEventCreate
                    0.02%  6.0009ms    

In [6]:
%%writefile matmul_cuda_tiled.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

// Allow the script to determine TILE_SIZE during compilation.
// use 16 as the default.
#ifndef TILE_SIZE
#define TILE_SIZE 16
#endif


__global__ void matMulKernel(double *a, double *b, double *c, int n) {
    __shared__ double As[TILE_SIZE][TILE_SIZE];
    __shared__ double Bs[TILE_SIZE][TILE_SIZE];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + tx;
    int row = blockIdx.y * TILE_SIZE + ty;

    double sum = 0.0;

    for (int t = 0; t < (n + TILE_SIZE - 1) / TILE_SIZE; t++) {
        if (row < n && t * TILE_SIZE + tx < n)
            As[ty][tx] = a[row * n + t * TILE_SIZE + tx];
        else
            As[ty][tx] = 0.0;

        if (t * TILE_SIZE + ty < n && col < n)
            Bs[ty][tx] = b[(t * TILE_SIZE + ty) * n + col];
        else
            Bs[ty][tx] = 0.0;

        __syncthreads();

        for (int k = 0; k < TILE_SIZE; k++) {
            sum += As[ty][k] * Bs[k][tx];
        }
        __syncthreads();
    }

    if (row < n && col < n) {
        c[row * n + col] = sum;
    }
}

int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N. Optional: use -d or --debug for output.\n");
        printf("Usage: %s <N> [-d]\n", argv[0]);
        return 1;
    }

    int n = 0;
    int debug_mode = 0;

    // Parse command line arguments
    for (int i = 1; i < argc; i++) {
        if (strcmp(argv[i], "-d") == 0 || strcmp(argv[i], "--debug") == 0) {
            debug_mode = 1;
        } else {
            n = atoi(argv[i]);
        }
    }

    if (n <= 0) {
        printf("Error: Invalid dimension N.\n");
        return 1;
    }
    size_t size = (size_t)n * n * sizeof(double);

    double *h_a = (double *)malloc(size);
    double *h_b = (double *)malloc(size);
    double *h_c = (double *)malloc(size);

    for (int i = 0; i < n * n; i++) {
        h_a[i] = 2.0; h_b[i] = 3.0; h_c[i] = 0.0;
    }

    cudaEvent_t start, stop_h2d, stop_kernel, stop_d2h;
    cudaEventCreate(&start); cudaEventCreate(&stop_h2d);
    cudaEventCreate(&stop_kernel); cudaEventCreate(&stop_d2h);

    double *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    cudaEventRecord(start);
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);
    cudaEventRecord(stop_h2d);

    // Use TILE_SIZE to define the block and grid dimensions
    dim3 threadsPerBlock(TILE_SIZE, TILE_SIZE);
    dim3 numBlocks((n + TILE_SIZE - 1) / TILE_SIZE, (n + TILE_SIZE - 1) / TILE_SIZE);

    matMulKernel<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaEventRecord(stop_kernel);

    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop_d2h);
    cudaEventSynchronize(stop_d2h);

    float ms_h2d, ms_kernel, ms_d2h;
    cudaEventElapsedTime(&ms_h2d, start, stop_h2d);
    cudaEventElapsedTime(&ms_kernel, stop_h2d, stop_kernel);
    cudaEventElapsedTime(&ms_d2h, stop_kernel, stop_d2h);

    printf("BlockSize: %dx%d\n", TILE_SIZE, TILE_SIZE);
    printf("H2D Time : %f ms\n", ms_h2d);
    printf("Kernel Time : %f ms\n", ms_kernel);
    printf("D2H Time : %f ms\n", ms_d2h);
      printf("Total Time (Data + Compute)  : %f ms\n", ms_h2d + ms_kernel + ms_d2h);

   // debug section
    if (debug_mode) {
        FILE *debug_file = fopen("debug_c.txt", "w");
        if (debug_file != NULL) {
            // Determine the print size, maximum 10x10 block
            int print_size = (n < 10) ? n : 10;

            fprintf(debug_file, "Debug Matrix C (N=%d)\n", n);
            fprintf(debug_file, "First %dx%d elements (top-left):\n", print_size, print_size);
            fprintf(debug_file, "--------------------------------------------------\n");

            for (int i = 0; i < print_size; i++) {
                for (int j = 0; j < print_size; j++) {
                    // %.2f prints 2 decimal places
                    fprintf(debug_file, "%10.2f ", h_c[i * n + j]);
                }
                fprintf(debug_file, "\n");
            }

            // Print the very last element to ensure the whole kernel worked
            fprintf(debug_file, "\nLast element C[%d][%d] = %.2f\n", n-1, n-1, h_c[(n-1)*n + (n-1)]);

            fclose(debug_file);
            printf("Debug ON: Output successfully saved to 'debug_c.txt'.\n");
        } else {
            printf("Debug ON: Error opening the debug file!\n");
        }
    }
    // ----------------------------------------------------------

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    cudaEventDestroy(start); cudaEventDestroy(stop_h2d);
    cudaEventDestroy(stop_kernel); cudaEventDestroy(stop_d2h);

    return 0;
}

Writing matmul_cuda_tiled.cu


In [7]:
!nvcc -arch=sm_75 -O3 matmul_cuda_tiled.cu -o matmul_gpu_tiled

In [ ]:
!/content/matmul_gpu_tiled 15000

BlockSize: 16x16
H2D Time : 820.836792 ms
Kernel Time : 33500.988281 ms
D2H Time : 381.752441 ms
Total Time (Data + Compute)  : 34703.578125 ms


In [ ]:
!nvprof /content/matmul_gpu_tiled 15000

==3029== NVPROF is profiling process 3029, command: /content/matmul_gpu_tiled 15000
BlockSize: 16x16
H2D Time : 837.021973 ms
Kernel Time : 33451.472656 ms
D2H Time : 394.462006 ms
Total Time (Data + Compute)  : 34682.957031 ms
==3029== Profiling application: /content/matmul_gpu_tiled 15000
==3029== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   96.45%  33.4508s         1  33.4508s  33.4508s  33.4508s  matMulKernel(double*, double*, double*, int)
                    2.41%  836.43ms         2  418.21ms  417.57ms  418.86ms  [CUDA memcpy HtoD]
                    1.14%  394.10ms         1  394.10ms  394.10ms  394.10ms  [CUDA memcpy DtoH]
      API calls:   99.43%  34.6822s         3  11.5607s  417.83ms  33.8453s  cudaMemcpy
                    0.54%  189.79ms         4  47.448ms  1.0540us  189.78ms  cudaEventCreate
                    0.02%  5.9718ms         3  1.9906ms  968.66us  2.8970ms  cudaFree
                  

In [ ]:
!find / -type f -name nsys 2>/dev/null

/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys


In [ ]:
!/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys profile -t cuda,osrt -o /content/report_matmul --stats=true --force-overwrite=true /content/matmul_gpu 10000

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
Matrix Dimension N           : 10000
Transfer Time CPU -> GPU     : 339.207520 ms
GPU Kernel Computation Time  : 9165.712891 ms
Transfer Time GPU -> CPU     : 184.270111 ms
Total Time (Data + Compute)  : 9689.190430 ms
Generating '/tmp/nsys-report-14f0.qdstrm'
[1/7] [========================100%] report_matmul.nsys-rep
[2/7] [========================100%] report_matmul.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  ----------------------
     98.1   10,211,829,690        111  91,998,465.7  100,131,715.0     1,817  292,234,959  34,126,013.2  poll                  
      1.9      195,553,605        535     36

In [ ]:
!/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys profile -t cuda,osrt -o /content/report_matmul_tiled --stats=true --force-overwrite=true /content/matmul_gpu_tiled 10000

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
BlockSize: 16x16
H2D Time : 351.576813 ms
Kernel Time : 8378.458008 ms
D2H Time : 184.059464 ms
Total Time (Data + Compute)  : 8914.094727 ms
Generating '/tmp/nsys-report-d326.qdstrm'
[1/7] [========================100%] report_matmul_tiled.nsys-rep
[2/7] [========================100%] report_matmul_tiled.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  ----------------------
     97.9    9,413,000,818        103  91,388,357.5  100,129,698.0     1,926  295,458,001  35,560,760.0  poll                  
      2.0      194,538,380        534     364,304.1       15,130.5     1,003   17,970,338   1,243,096.1  ioct

In [ ]:
%%bash

set -euo pipefail

RESULTS_DIR="./results"
mkdir -p "$RESULTS_DIR"

CSV_FILE="$RESULTS_DIR/cuda_scalability.csv"
RAW_LOG="$RESULTS_DIR/cuda_raw.log"

echo "N,BlockSize_1D,ThreadsPerBlock,H2D_ms,Kernel_ms,D2H_ms" > "$CSV_FILE"
> "$RAW_LOG"

echo "=================================================="
echo " STARTING CUDA BENCHMARK"
echo "=================================================="

# Choose matrix dimensions
SIZES=(5000 10000 15000)

# 2D Block configurations (TILE_SIZE)
BLOCK_SIZES=(8 16 32)

for N in "${SIZES[@]}"; do
    echo ""
    echo "--- Testing Matrix Size N = $N ---"

    for B in "${BLOCK_SIZES[@]}"; do
        TOTAL_THREADS=$((B * B))
        printf "Compiling and Executing Block %2dx%-2d (%4d threads) ... " "$B" "$B" "$TOTAL_THREADS"

        # Pass $B to the compiler. It will save the temporary executable as matmul_gpu_test
        nvcc -arch=sm_75 matmul_cuda_tiled.cu -DTILE_SIZE=$B -O3 -o /content/matmul_gpu_test

        #Execute the newly forged executable for this dimension
        OUTPUT=$(/content/matmul_gpu_test "$N")

        # Save the complete output for debugging
        echo "=== N=$N, Block=${B}x${B} ===" >> "$RAW_LOG"
        echo "$OUTPUT" >> "$RAW_LOG"

        # Extract times using grep and awk
        TIME_H2D=$(echo "$OUTPUT" | grep "H2D Time" | awk '{print $(NF-1)}')
        TIME_KER=$(echo "$OUTPUT" | grep "Kernel Time" | awk '{print $(NF-1)}')
        TIME_D2H=$(echo "$OUTPUT" | grep "D2H Time" | awk '{print $(NF-1)}')

        echo "Kernel: ${TIME_KER} ms"

        # Write to CSV
        echo "$N,$B,$TOTAL_THREADS,$TIME_H2D,$TIME_KER,$TIME_D2H" >> "$CSV_FILE"
    done
done

echo "=================================================="
echo " BENCHMARK COMPLETED!"
echo " Results saved in: $CSV_FILE"

 STARTING CUDA BENCHMARK

--- Testing Matrix Size N = 5000 ---
Compiling and Executing Block  8x8  (  64 threads) ... Kernel: 1277.603027 ms
Compiling and Executing Block 16x16 ( 256 threads) ... Kernel: 1132.626709 ms
Compiling and Executing Block 32x32 (1024 threads) ... Kernel: 1140.363770 ms

--- Testing Matrix Size N = 10000 ---
Compiling and Executing Block  8x8  (  64 threads) ... Kernel: 10050.465820 ms
Compiling and Executing Block 16x16 ( 256 threads) ... Kernel: 8566.209961 ms
Compiling and Executing Block 32x32 (1024 threads) ... Kernel: 8726.650391 ms

--- Testing Matrix Size N = 15000 ---
Compiling and Executing Block  8x8  (  64 threads) ... Kernel: 38472.062500 ms
Compiling and Executing Block 16x16 ( 256 threads) ... Kernel: 33504.539062 ms
Compiling and Executing Block 32x32 (1024 threads) ... Kernel: 30064.720703 ms
 BENCHMARK COMPLETED!
 Results saved in: ./results/cuda_scalability.csv


In [3]:
!ncu --set full -o prof_naive /content/matmul_gpu 5000

==PROF== Connected to process 1512 (/content/matmul_gpu)
==PROF== Profiling "matMulKernel" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Matrix Dimension N           : 5000
Transfer Time CPU -> GPU     : 94.284195 ms
GPU Kernel Computation Time  : 209775.000000 ms
Transfer Time GPU -> CPU     : 62.844128 ms
Total Time (Data + Compute)  : 209932.125000 ms
==PROF== Disconnected from process 1512
==PROF== Report: /content/prof_naive.ncu-rep


In [4]:
!nvcc -arch=sm_75 matmul_cuda_tiled.cu -DTILE_SIZE=16 -O3 -o /content/matmul_gpu_tiled

cc1plus: fatal error: matmul_cuda_tiled.cu: No such file or directory
compilation terminated.


In [10]:
!ncu --set full -o prof_tiled_16x16 /content/matmul_gpu_tiled 5000

==PROF== Connected to process 6652 (/content/matmul_gpu_tiled)
==PROF== Profiling "matMulKernel" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
BlockSize: 16x16
H2D Time : 91.705917 ms
Kernel Time : 196035.968750 ms
D2H Time : 50.051872 ms
Total Time (Data + Compute)  : 196177.718750 ms
==PROF== Disconnected from process 6652
==PROF== Report: /content/prof_tiled_16x16.ncu-rep
